# 02. Прогноз общего объёма
Подключите распакованный results.zip из 00_baseline. Ноутбук использует CPU. Для каждого из восьми горизонтов Ridge прогнозирует отношение сетевого объёма к среднему объёму шести последних известных получасовых точек. Все признаки доступны в 10:30. Коэффициент согласования маршрутных прогнозов выбирается на 4–17 октября. Значение 0 сохраняет контроль.

Для подбора модель общего объёма обучается на истории до 4 октября. Для проверки она переобучается до 18 октября и фиксируется на обе недели. Маршрутные прогнозы имеют исходный weekly-протокол. Контрольные маршрутные модели обновляются каждую неделю, поэтому при прогнозе второй недели им уже доступны факты первой.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

roots = [Path("matvey/results-00_baseline"), Path("results-00_baseline"), Path("/kaggle/input")]
paths = [p for root in roots if root.is_dir() for p in root.rglob("weekly_predictions.parquet")]
if len(paths) != 1:
    raise ValueError(f"Нужен один распакованный baseline_00, найдено: {paths}")
root = paths[0].parent
weekly = pd.read_parquet(paths[0]).query("period != 'warmup'").copy()
full = pd.read_parquet(root / "full_predictions.parquet")
history = pd.read_parquet(root / "network_history.parquet").set_index("timestamp").sort_index()
target = history.target_1h
step = pd.Timedelta(minutes=30)
days = target.index[(target.index.hour == 10) & (target.index.minute == 30)]
test_start = weekly.loc[weekly.period.eq("holdout"), "cutoff"].min()
tune_start = weekly.loc[weekly.period.eq("tuning"), "cutoff"].min()
if not (weekly.model_info_through < weekly.cutoff).all():
    raise ValueError("Нарушена временная граница контроля")
if full.cutoff.nunique() != 1 or full.cutoff.iloc[0] != days[-1]:
    raise ValueError("Проверьте дату финального прогноза")

In [ ]:
def features(h):
    rows = []
    for day in days:
        morning = history.loc[day-5*step:day]
        if len(morning) < 6 or day-pd.Timedelta(days=28)-5*step < history.index[0]:
            continue
        level = morning.target_1h.mean()
        typical = []
        for lag in (1, 2, 7, 14, 21, 28):
            start = day-pd.Timedelta(days=lag)
            usual = target.loc[start-5*step:start].mean()
            typical.append(target.loc[start+h*step] / max(usual, 1))
        values = {
            "slope": morning.target_1h.iloc[-1] / max(level, 1),
            "recent": morning.target_1h.iloc[-2:].mean() / max(level, 1),
            "day1": typical[0], "day2": typical[1], "weekday_profile": np.mean(typical[2:]),
            "weekday_sin": np.sin(2*np.pi*day.dayofweek/7),
            "weekday_cos": np.cos(2*np.pi*day.dayofweek/7),
            "saturday": float(day.dayofweek == 5), "sunday": float(day.dayofweek == 6),
        }
        for name in history.columns.drop("target_1h"):
            values[name] = morning[name].iloc[-1] / max(level, 1)
        rows.append(dict(cutoff=day, volume=target.get(day+h*step, np.nan), level=level, **values))
    return pd.DataFrame(rows).set_index("cutoff")

def metrics(y, p, days):
    delta = np.asarray(p)-np.asarray(y)
    sums = pd.DataFrame(dict(day=np.asarray(days), y=np.asarray(y), absolute=np.abs(delta), signed=delta)).groupby("day").sum()
    total = sums.y.sum()
    return dict(WAPE=sums.absolute.sum()/total, bias=sums.signed.sum()/total,
                score=(sums.absolute.sum()+abs(sums.signed.sum()))/total,
                daily_score=((sums.absolute+sums.signed.abs())/sums.y).mean())

holdout_reports = {}

def fit_total(h, boundary, dates, period):
    table = features(h)
    columns = table.columns.drop(["volume","level"])
    train = table.loc[(table.index + h*step < boundary.normalize()) & table.volume.notna()]
    query = table.loc[dates]
    model = make_pipeline(StandardScaler(), Ridge(alpha=10))
    model.fit(train[columns], train.volume/train.level)
    def predict(rows):
        return np.maximum(0, model.predict(rows[columns]) * rows.level)
    pred = predict(query)
    report = [dict(period="train", **metrics(train.volume, predict(train), train.index))]
    if period != "future":
        report.append(dict(period=period, **metrics(query.volume, pred, query.index)))
    if period == "holdout":
        holdout_reports[h] = report[-1].copy()
    if period == "future":
        import joblib
        path = Path("volume_models")
        path.mkdir(exist_ok=True)
        joblib.dump(model,path/f"h{h}.joblib")
        report.append(dict(holdout_reports[h], period="test до refit"))
    display(pd.DataFrame(report).assign(horizon=h))
    return pd.DataFrame(dict(cutoff=dates, horizon=h, total=pred.to_numpy()))

In [ ]:
def attach(frame, totals):
    result = frame.merge(totals, on=["cutoff","horizon"], how="left", validate="many_to_one")
    group = result.groupby(["cutoff","horizon"])
    if not group.size().eq(1000).all() or result.duplicated(["cutoff","horizon","route_id"]).any():
        raise ValueError("Неполная панель маршрутов")
    base = group.y_pred.transform("sum")
    if base.le(0).any() or not np.isfinite(result[["total","y_pred"]]).all().all():
        raise ValueError("Недопустимый сетевой прогноз")
    result["ratio"] = result.total / base
    return result

def reconciled(frame, weight):
    return frame.y_pred * (1 + weight * (frame.ratio - 1))

def evaluate(period, boundary):
    panel = weekly.loc[weekly.period.eq(period)]
    dates = pd.DatetimeIndex(panel.cutoff.unique()).sort_values()
    totals = pd.concat([fit_total(h, boundary, dates, period) for h in range(1,9)])
    return attach(panel, totals)

def summary(frame, weight):
    return dict(weight=weight, **metrics(frame.y, reconciled(frame, weight), frame.cutoff))

tuning = evaluate("tuning", tune_start)
scores = pd.DataFrame([summary(tuning, w) for w in [0, 0.25, 0.5, 0.75, 1]])
weight = float(scores.loc[scores.daily_score.idxmin(), "weight"])
display(scores)
print("Вес зафиксирован на подборе:", weight)
holdout = evaluate("holdout", test_start)
rows = []
for period, frame in [("tuning", tuning), ("test", holdout)]:
    for group, part in [("все даты", frame), ("субботы", frame.loc[frame.cutoff.dt.dayofweek.eq(5)])]:
        for w in sorted({0., weight}):
            rows.append(dict(period=period, group=group, **summary(part,w)))
report = pd.DataFrame(rows)
display(report)
total_panel = pd.concat([tuning, holdout])[["cutoff","horizon","total"]].drop_duplicates()
total_panel.to_parquet("volume_history.parquet", index=False)
report.to_csv("total_volume_report.csv", index=False)
print("Оценка test получена до финального переобучения. Даты локального test ранее изучались.")

## Финальный прогноз
При положительном весе восемь моделей общего объёма переобучаются на всех доступных размеченных датах с полной историей признаков. Маршрутный компонент берётся из полного refit этапа 00. Перед переносом поправки обратите внимание на различие калибровки weekly и финального контроля.

In [ ]:
future = full.copy()
if weight > 0:
    dates = pd.DatetimeIndex(full.cutoff.unique())
    totals = pd.concat([fit_total(h, dates[0].normalize()+pd.Timedelta(days=1), dates, "future") for h in range(1,9)])
    totals.to_parquet("volume_future.parquet", index=False)
    future = attach(full, totals)
    future["y_pred"] = reconciled(future, weight)
control_ids = pd.read_csv(root / "submission.csv", usecols=["id"])
submission = control_ids.merge(future[["id","y_pred"]], on="id", how="left", validate="one_to_one")
if len(submission) != 8000 or not np.isfinite(submission.y_pred).all() or submission.y_pred.lt(0).any():
    raise ValueError("Проверьте полноту submission")
submission.to_csv("submission_total_volume.csv", index=False)
import json
Path("volume_settings.json").write_text(json.dumps(dict(weight=weight, run_id=weekly.run_id.unique().item()), indent=2))